In [1]:
import numpy as np
import pandas as pd
import h5py
import os
import json
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from numpy.lib.stride_tricks import sliding_window_view
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from scipy.spatial.distance import pdist
from scipy.stats import zscore
import tensorflow as tf
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Dense, Conv1D, BatchNormalization, Bidirectional, LSTM, Concatenate, Activation, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import Huber, MeanSquaredError, LogCosh, MeanAbsoluteError
from tensorflow.keras.initializers import GlorotUniform, Orthogonal

from utils2 import random_extraction, extract_hrv_features, fast_predict, evaluate_imputation_performance

ages_table_path = 'ages_table.xlsx'
ages_table = pd.read_excel(ages_table_path)

# Updated feature columns (12 static features)
feature_cols = [
    'rmssd', 'rs', 'ccm', 'ccm_n5', 'guzik', 'nn20', 'porta',
    'ar_1', 'ar_2', 'ar_3', 'ar_4', 'ar_5'
]

# 30 RR intervals
rr_cols = [f"rr_{i}" for i in range(1, 31)]


def canonical_code(x):
    s = str(x).strip()
    if s.isdigit():
        return f"{int(s):03d}" if len(s) < 3 else s
    return s


def get_subject_interval(h5_path, subject):
    subject_key = canonical_code(subject)

    with h5py.File(h5_path, "r") as h5:
        subject_ids = [
            x.decode("utf-8") if isinstance(x, (bytes, np.bytes_)) else str(x)
            for x in h5["index"]["subject_id"][()]
        ]
        intervals = [
            x.decode("utf-8") if isinstance(x, (bytes, np.bytes_)) else str(x)
            for x in h5["index"]["interval"][()]
        ]

        for sid, interval in zip(subject_ids, intervals):
            if canonical_code(sid) == subject_key:
                return interval

    raise ValueError(f"Subject {subject} not found in {h5_path}")


def load_normalization_params(
    h5_path,
    interval=None,   # Kept for backward compatibility with your loops
    age_years=None,  # Kept for backward compatibility with your loops
):
    """
    Returns:
        feats_mean : (12,)
        feats_scale : (12,)
        seq_mean : scalar
        seq_scale : scalar
        y_mean : scalar
        y_scale : scalar
    """
    with h5py.File(h5_path, "r") as h5:
        # Load from the global normalization group
        norm_grp = h5["normalization"]

        cols = [
            c.decode("utf-8") if isinstance(c, (bytes, np.bytes_)) else str(c)
            for c in norm_grp["columns"][:]
        ]

        mean = norm_grp["mean"][:]
        std = norm_grp["std"][:]
        
        # ----------------------------------
        # Target & Sequence (Global Attributes)
        # ----------------------------------
        seq_mean = float(norm_grp.attrs["rr_mean"])
        seq_scale = float(norm_grp.attrs["rr_std"])
        
        y_mean = float(norm_grp.attrs["target_mean"])
        y_scale = float(norm_grp.attrs["target_std"])

    # Map column names to their mean and std
    stats = {
        col: (m, s)
        for col, m, s in zip(cols, mean, std)
    }

    # ----------------------------------
    # Static features
    # ----------------------------------
    # Dynamically extract means and scales matching the feature_cols layout exactly
    feats_mean = np.array([stats[c][0] for c in feature_cols], dtype=np.float32)
    feats_scale = np.array([stats[c][1] for c in feature_cols], dtype=np.float32)

    return (
        feats_mean,
        feats_scale,
        seq_mean,
        seq_scale,
        y_mean,
        y_scale,
    )

I0000 00:00:1791309815.661998 1534707 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1791309817.793470 1534707 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
class RSAPhaseAwareLoss(tf.keras.losses.Loss):
    # Added **kwargs to accept 'reduction' and other base class args
    def __init__(self, delta=1.0, lambda_sign=0.2, lambda_var=0.1, name="rsa_phase_loss", **kwargs):
        super().__init__(name=name, **kwargs)
        self.delta = delta
        self.lambda_sign = lambda_sign
        self.lambda_var = lambda_var
        self.huber = tf.keras.losses.Huber(delta=self.delta)

    def call(self, y_true, y_pred):
        huber_loss = self.huber(y_true, y_pred)
        # Penaliza predicciones en contrafase respecto a la pendiente respiratoria
        sign_penalty = tf.reduce_mean(tf.nn.relu(-y_true * y_pred))
        # Preserva la amplitud y dispersión oscilatoria
        var_true = tf.math.reduce_variance(y_true)
        var_pred = tf.math.reduce_variance(y_pred)
        var_penalty = tf.abs(var_true - var_pred)

        return huber_loss + self.lambda_sign * sign_penalty + self.lambda_var * var_penalty

    # Added get_config so Keras knows how to serialize and deserialize the custom parameters
    def get_config(self):
        config = super().get_config()
        config.update({
            "delta": self.delta,
            "lambda_sign": self.lambda_sign,
            "lambda_var": self.lambda_var,
        })
        return config

# Ensure the custom loss is defined or imported in this notebook/script
# from your_model_script import RSAPhaseAwareLoss 
# (If it's defined in the same notebook, Keras will find it via custom_objects below)

model_path = 'tcn_attention_hrv_best.keras'
# We must register the custom loss function to load the saved model successfully
loaded_model = load_model(
    model_path, 
    custom_objects={'RSAPhaseAwareLoss': RSAPhaseAwareLoss}
)

series_path = 'series/'
subjects_test = [
    # "nsr010RRcl.txt",
    # "nsr038RRcl.txt",
    "nsr041RRcl.txt",
    "nsr013RRcl.txt",
    "nsr033RRcl.txt"
]

# Create your array of percentages
percents_array = np.array([0.8])

with open("ar_model_parameters.json", "r") as f:
    ar_parameters_by_range = json.load(f)

range_key = f"{3220}-{3481}"
params = ar_parameters_by_range[range_key]

phi_burg = np.array(params["phi_burg"], dtype=float)
sigma_burg = float(params["sigma_burg"])

# ---------------------------------------------------------
# LOAD GLOBAL NORMALIZATION PARAMETERS ONCE (Outside the loop)
# ---------------------------------------------------------
(
    feats_mean,
    feats_scale,
    seq_mean,
    seq_scale,
    y_mean,
    y_scale,
) = load_normalization_params("hrv_dataset.h5")


for subject in subjects_test:

    original_serie = np.loadtxt(os.path.join(series_path, subject), dtype=int).astype(float)
    print(f"Processing: {os.path.join(series_path, subject)}")
    
    subject_name = os.path.splitext(subject)[0]
    folder = Path(subject_name)
    folder.mkdir(parents=True, exist_ok=True)

    # Note: get_subject_interval() and age_years calculations were completely 
    # removed here because global normalization no longer requires them!

    # Run the function
    _ = evaluate_imputation_performance(
        original_serie=original_serie,
        percents_to_eliminate=percents_array,
        loaded_model=loaded_model,
        feats_mean=feats_mean, 
        feats_scale=feats_scale,
        seq_mean=seq_mean, 
        seq_scale=seq_scale,
        y_mean=y_mean, 
        y_scale=y_scale,
        feature_cols=feature_cols, 
        rr_cols=rr_cols,
        path_to_save=subject_name,
        subject_name=subject_name,
        phi_burg=phi_burg, 
        sigma_burg=sigma_burg, 
        ages_table=ages_table
    )

E0000 00:00:1791309819.693907 1534707 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1791309819.694313 1534784 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
E0000 00:00:1791309819.706313 1534707 cuda_executor.cc:1827] Nvml call failed with 3(Not Supported). Assuming PCIe gen 3 x16 bandwidth.
W0000 00:00:1791309819.706916 1534707 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Processing: series/nsr041RRcl.txt
Starting autoregressive imputation across 1 thresholds and 1 seeds using 4 cores...
✅ Finished: Seed 7 | Elimination: percent_80%
🎉 All parallel tasks completed successfully!
Processing: series/nsr013RRcl.txt
Starting autoregressive imputation across 1 thresholds and 1 seeds using 4 cores...
✅ Finished: Seed 7 | Elimination: percent_80%
🎉 All parallel tasks completed successfully!
Processing: series/nsr033RRcl.txt
Starting autoregressive imputation across 1 thresholds and 1 seeds using 4 cores...
✅ Finished: Seed 7 | Elimination: percent_80%
🎉 All parallel tasks completed successfully!
